## 進化動学

**ゲーム** : このゲームでは店側と客側の2つのプレイヤーが存在します。プレイヤーは選択肢 (0, 1) があり選択肢によってそれぞれ利得が決まる。

# 利得表
# 利得は (店の利得, 客の利得) で表される。
# |         | 店: O | 店: I |
# |---------|-------|-------|
# | 客: O    |  1, 1  |  0, -1  |
# | 客: I    |  -1, 0  |  3, 3  |

しかし、実社会では沢山の店があり、沢山の客がいます。そこで、進化動学を用いてこのゲームを解析します。
店、客をエージェントとし、その集団をプレイヤーとしてゲームを分析する。（混合戦略）

## 混合戦略
**戦略**
各プレイヤーの戦略は各エージェントが取りえる手の割合で表現される。
例えば、店の戦略がx=0.2であれば、店のエージェントの20%が選択肢Oを選び、80%が選択肢Iを選ぶことを意味する。
**利得**
利得は各エージェントの利得の平均で計算される。

## 反復ゲーム
このゲームを反復的に行い、各プレイヤーの戦略が時間とともにどのように変化するかを観察します。
現実は各エージェントは利得の最大化の戦略は特定できません。そのため、現実では大衆の行動に基づいて戦略を更新します。流行に乗ってものを選ぶことがあります。（大衆戦略の模倣）。
進化動学では、各エージェントは自分の利得と大衆の利得を比較し、自分の利得が低ければ大衆の戦略を模倣します。これにより、各プレイヤーの戦略は時間とともに変化します。（反復ゲーム）

### エージェントの作成

In [ ]:
class PlayerRole():
    """
    strgSet: 戦略の集合 list[num] (例: [0, 1])
    payMat: 利得行列 list[list] n次元の配列, nは戦略の数 (例: n=2の場合[[1, 0], [-1, 3]]) 
    nmStrg: 戦略の数 int (例: 2)
    revProt: ゲームの更新のための関数
    """
    def __init__(self, strgSet, payMat, revProt):
        self.strgSet = strgSet
        self.payMat = payMat
        self.nmStrg = len(self.strgSet)
        self.revProt = revProt
    
    def expPayFn(self, strgDist_opp):
        # 相手（opp = opponent）
        """
        strgDist_opp: 相手の戦略分布 list[num] (例: [0.7, 0.3] -> 相手が戦略0をとる確率が70%, 戦略1をとる確率が30%)
        """
        # 期待利得のベクトルの初期化
        payVec = [0 for i in range(self.nmStrg)] 
        # 期待利得の計算
        for strgId_Id in range(self.nmStrg):
            for strgId_opp in range(len(strgDist_opp)):
                payVec[strgId_Id] += self.payMat[strgId_Id][strgId_opp] * strgDist_opp[strgId_opp]
        return payVec

In [8]:
## 例: 2戦略のゲーム
# プレイヤー1
# 戦略の集合
strgSet_1 = [0, 1]
# 利得行列
payMat_1 = [[1, 0], [-1, 3]]
player1 = PlayerRole(strgSet_1, payMat_1, None)
# プレイヤー2
# 戦略の集合
strgSet_2 = [0, 1]
# 利得行列
payMat_2 = [[-1, 1], [0, -3]]
player2 = PlayerRole(strgSet_2, payMat_2, None)

# 利得表
"""
|                  | Player 2: Strg 0 | Player 2: Strg 1 |
|------------------|------------------|------------------|
| Player 1: Strg 0 | (1, -1)          | (0, 0)           |
| Player 1: Strg 1 | (-1, 1)          | (3, -3)          |

"""

# プレイヤー1の戦略分布
strgDist_1 = [0.7, 0.3]
# プレイヤー2の戦略分布
strgDist_2 = [0.4, 0.6]
# プレイヤー1の期待利得の計算
payVec_1 = player1.expPayFn(strgDist_2)
print("Player 1's expected payoff vector:", payVec_1)
# プレイヤー2の期待利得の計算
payVec_2 = player2.expPayFn(strgDist_1)
print("Player 2's expected payoff vector:", payVec_2)

# 計算式
"""
Player 1's Strg payoff vector:
    strg 0: 1 * 0.4 + 0 * 0.6 = 0.4
    strg 1: -1 * 0.4 + 3 * 0.6 = 1.4
Player 2's Strg payoff vector:
    strg 0: -1 * 0.7 + 1 * 0.3 = -0.4
    strg 1: 0 * 0.7 + (-3) * 0.3 = 0.9
"""

1つ目：1, 2つ目：0.4
payVec[0] = 0.4
1つ目：0, 2つ目：0.6
payVec[0] = 0.4
1つ目：-1, 2つ目：0.4
payVec[1] = -0.4
1つ目：3, 2つ目：0.6
payVec[1] = 1.4
Player 1's expected payoff vector: [0.4, 1.4]
1つ目：-1, 2つ目：0.7
payVec[0] = -0.7
1つ目：1, 2つ目：0.3
payVec[0] = -0.39999999999999997
1つ目：0, 2つ目：0.7
payVec[1] = 0.0
1つ目：-3, 2つ目：0.3
payVec[1] = -0.8999999999999999
Player 2's expected payoff vector: [-0.39999999999999997, -0.8999999999999999]


##  進化動学: 利得を比較して戦略を更新する
スイッチングレート：プレイヤーが戦略を更新しやすさを表すパラメータ。スイッチングレートが高いほど、プレイヤーは戦略を更新する傾向がある。

In [ ]:
# Ios: Imitating of Success（成功の模倣）
class IoS():
    def __init__(self, payAdd = 0):
        self.payAdd = payAdd
    def __call__(self, payVec, strgDist_own):
        strgIdSet = range(len(payVec))
        maxPay=max([payVec[strgId] for strgId in strgIdSet])
        switchMtrx = [[0 for i in strgIdSet] for j in strgIdSet]

        for strgIdForm in strgIdSet:
            tempVec = [0 for strgIdTo in strgIdSet]
            for strgIdTo in [strgId for strgId in strgIdSet if strgId != strgIdForm]:
                tempVec[strgIdTo] = strgDist_own[strgIdTo] * (payVec[strgIdTo] + self.payAdd) / (maxPay + self.payAdd)
            switchMtrx[strgIdForm] = tempVec
            switchMtrx[strgIdForm][strgIdForm] = 1 - sum(tempVec)
        return switchMtrx

# BRD（最適反応動学）でのスイッチングレート
def stdBRD(payVec, strgDist_own):
    strgIdSet = range(len(payVec))
    maxPay=max([payVec[strgId] for strgId in strgIdSet])
    switchMtrx = [[0 for i in strgIdSet] for j in strgIdSet]

    maxStrgIndx=[strgId for strgId in strgIdSet if payVec[strgId]==maxPay]
    for strgIdFrom in strgIdSet:
        for strgIdTo in maxStrgIndx:
            switchMtrx[strgIdFrom][strgIdTo] = 1/len(maxStrgIndx)
    return switchMtrx

def tBRD(payVec, strgDist_own, maxPayDiff):
    strgIdSet = range(len(payVec))
    switchMtrx = [ [ 0 for strgIdTo in strgIdSet] for strgIdFrom in strgIdSet]
    maxPay=max([payVec[strgId] for strgId in strgIdSet])
    
    maxStrgId=[strgId for strgId in strgIdSet if payVec[strgId]==maxPay]
    for strgIdFrom in strgIdSet:
        tempVec=[ 0 for strgIdTo in strgIdSet]
        for strgIdTo in [strgId for strgId in maxStrgId if strgId !=strgIdFrom]:
            tempVec[strgIdTo] = (maxPay - payVec[strgIdFrom])/(maxPayDiff*len(maxStrgId))
        switchMtrx[strgIdFrom] = tempVec
        switchMtrx[strgIdFrom][strgIdFrom] = 1- sum(tempVec)
    return switchMtrx

In [ ]:
class RevisionProt():
    def __init__(self, payVec, payAdd = 0):
        self.payVec = payVec
        self.payAdd = payAdd

        self.strgIdSet = range(len(payVec)) # 期待利得のベクトル次元数
        self.maxPay = max([payVec[strgId] for strgId in self.strgIdSet]) # 期待利得の最大値
        self.switchMtrx = [[0 for i in self.strgIdSet] for j in self.strgIdSet] # スイッチングレートの行列、戦略変更の確率を表す行列

    
    # IoS: Imitating of Success（成功の模倣）
    def IoS(self, strgDist_own):
        for strgIdForm in self.strgIdSet:
            tempVec = [0 for strgIdTo in self.strgIdSet]
            for strgIdTo in [strgId for strgId in self.strgIdSet if strgId != strgIdForm]:
                tempVec[strgIdTo] = strgDist_own[strgIdTo] * (self.payVec[strgIdTo] + self.payAdd) / (self.maxPay + self.payAdd)
            self.switchMtrx[strgIdForm] = tempVec
            self.switchMtrx[strgIdForm][strgIdForm] = 1 - sum(tempVec)
        return self.switchMtrx

    # stdBRD: Best Response Dynamics（最適反応動学）へのスイッチ
    def stdBRD(self, strgDist_own):
        maxStrgId=[strgId for strgId in self.strgIdSet if self.payVec[strgId]==self.maxPay]
        for strgIdFrom in self.strgIdSet:
            for strgIdTo in maxStrgId:
                self.switchMtrx[strgIdFrom][strgIdTo] = 1/len(maxStrgId)
        return self.switchMtrx

    # tBRD: Temperature BRD（温度付き最適反応動学）へのスイッチ、緩やかに最適反応動学に近づく
    def tBRD(self, strgDist_own, maxPayDiff):
        maxStrgId=[strgId for strgId in self.strgIdSet if self.payVec[strgId]==self.maxPay]
        for strgIdFrom in self.strgIdSet:
            tempVec=[ 0 for strgIdTo in self.strgIdSet]
            for strgIdTo in [strgId for strgId in maxStrgId if strgId !=strgIdFrom]:
                tempVec[strgIdTo] = (self.maxPay - self.payVec[strgIdFrom]) / (maxPayDiff * len(maxStrgId))
            self.switchMtrx[strgIdFrom] = tempVec
            self.switchMtrx[strgIdFrom][strgIdFrom] = 1- sum(tempVec)
        return self.switchMtrx

In [ ]:
dyn = RevisionProt(payVec_1)

## 各プレイヤーの戦略更新のシミュレーション